# 13 Python intermediate - `functools`  v2.0

## Rozkład jazdy

1. 🔹 **`functools.partial`** - częściowe aplikowanie funkcji
2. 🔹 **`functools.lru_cache` i memoizacja** - cache wyników
3. 🔹 **`functools.reduce` i `total_ordering`** - redukcja i porównania
   🐍 Ćwiczenia do każdej sekcji

---

## 1. 🔹 `functools.partial`

`functools.partial(func, *args, **kwargs)` tworzy nową funkcję,
w której część argumentów jest już ustalona (zamrożona).
Wynik to obiekt `partial`, który można wywoływać jak zwykłą funkcję.

Przydaje się gdy:
- chcemy dostosować funkcję biblioteczną bez jej modyfikacji,
- potrzebujemy funkcji z mniejszą liczbą parametrów (np. jako
  callback),
- tworzymy specjalizowane wersje ogólnej funkcji.

> 💡 `partial` różni się od lambda tym, że zachowuje oryginalną
> sygnaturę funkcji w `__wrapped__` i lepiej współgra z debuggerami.

In [ ]:
from functools import partial


def power(base: float, exp: float) -> float:
    return base ** exp


square = partial(power, exp=2)
cube   = partial(power, exp=3)

print(square(4))   # 16.0
print(cube(3))     # 27.0


# partial z wbudowanymi funkcjami
from functools import partial

int_base2 = partial(int, base=2)   # int z base=2
print(int_base2('1010'))    # 10
print(int_base2('11111'))   # 31


# partial jako callback
def apply(func, values: list) -> list:
    return [func(v) for v in values]


add10 = partial(lambda x, n: x + n, n=10)
print(apply(add10, [1, 2, 3]))   # [11, 12, 13]

---

### 🐍 Ćwiczenia - partial

**Ćw. 1.** Utwórz funkcję `greet(name, greeting='Hello')` i za
pomocą `partial` stwórz `greet_pl` z polskim powitaniem `'Cześć'`.
Wywołaj obie wersje.

**Ćw. 2.** Masz funkcję `log(level, message, timestamp)`. Użyj
`partial`, aby stworzyć gotowe loggery: `log_info` i `log_error`
(z ustalonymi `level` i `timestamp=False`).

**Ćw. 3.** *(Trudniejsze)* Użyj `partial` i `map()`, aby
przetransformować listę słowników - ustaw klucz `'active'` na
`True` dla każdego elementu. Zdefiniuj funkcję pomocniczą
`set_field(d, key, value)` i stwórz jej partial.

In [ ]:
# Ćw. 1
from functools import partial

def greet(name: str, greeting: str = 'Hello') -> str:
    return f'{greeting}, {name}!'


greet_pl = ...
print(greet('Alice'))
print(greet_pl('Alice'))

In [ ]:
# Ćw. 2
from functools import partial
from datetime import datetime

def log(level: str, message: str, timestamp: bool = True) -> str:
    ts = f'[{datetime.now():%H:%M:%S}] ' if timestamp else ''
    return f'{ts}[{level}] {message}'


log_info  = ...
log_error = ...

print(log_info('Server started'))
print(log_error('Connection refused'))

In [ ]:
# Ćw. 3
from functools import partial

def set_field(d: dict, key: str, value) -> dict:
    return {**d, key: value}


users = [
    {'name': 'Alice', 'active': False},
    {'name': 'Bob',   'active': False},
]
activate = ...
result = list(map(activate, users))
print(result)

---

## 2. 🔹 `functools.lru_cache` i memoizacja

Memoizacja (memoization) to technika cachowania wyników funkcji,
żeby nie obliczać ich ponownie dla tych samych argumentów.

`@lru_cache(maxsize=128)` automatycznie cachuje wyniki funkcji.
LRU (Least Recently Used) oznacza, że gdy cache jest pełny,
usuwane są najrzadziej używane wpisy.

| Dekorator | Python | Opis |
|---|---|---|
| `@lru_cache(maxsize=N)` | 3.2+ | cache LRU, N wpisów |
| `@lru_cache(maxsize=None)` | 3.2+ | nieograniczony cache |
| `@cache` | 3.9+ | skrót dla `lru_cache(maxsize=None)` |

> 💡 Funkcja z `@lru_cache` musi przyjmować tylko hashowalne
> argumenty (nie może przyjmować list ani słowników jako parametrów).

In [ ]:
from functools import lru_cache
import time


# bez cache - wykładnicza złożoność
def fib_slow(n: int) -> int:
    if n < 2:
        return n
    return fib_slow(n-1) + fib_slow(n-2)


@lru_cache(maxsize=None)
def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)


t0 = time.perf_counter()
print(fib(35))
print(f'cached:   {time.perf_counter()-t0:.6f}s')

t0 = time.perf_counter()
print(fib_slow(35))
print(f'no cache: {time.perf_counter()-t0:.6f}s')

# statystyki cache
print(fib.cache_info())

---

### 🐍 Ćwiczenia - lru_cache

**Ćw. 4.** Napisz funkcję `count_ways(n)`, która oblicza ile
sposobów jest na wejście po schodach do n-tego stopnia, jeśli
można stawiać 1 lub 2 kroki naraz. Zastosuj `@lru_cache`.

**Ćw. 5.** Napisz funkcję `expensive_computation(x, y)` symulującą
kosztowną operację przez `time.sleep(0.1)`. Ozdób ją `@lru_cache`.
Zmierz czas pierwszego i kolejnego wywołania z tymi samymi args.

**Ćw. 6.** *(Trudniejsze)* Napisz funkcję `memoize(func)` - własny
dekorator memoizacji bez użycia `lru_cache`. Przechowuj wyniki
w słowniku `cache`. Sprawdź poprawność na funkcji Fibonacciego.
# hint: użyj args jako klucza słownika

In [ ]:
# Ćw. 4
from functools import lru_cache

@lru_cache(maxsize=None)
def count_ways(n: int) -> int:
    ...


for i in range(1, 8):
    print(f'count_ways({i}) = {count_ways(i)}')
# 1,2,3,5,8,13,21

In [ ]:
# Ćw. 5
from functools import lru_cache
import time

@lru_cache(maxsize=128)
def expensive_computation(x: int, y: int) -> int:
    time.sleep(0.1)
    return x * y + x - y


t0 = time.perf_counter()
print(expensive_computation(10, 20))
print(f'first call:  {time.perf_counter()-t0:.4f}s')

t0 = time.perf_counter()
print(expensive_computation(10, 20))
print(f'second call: {time.perf_counter()-t0:.4f}s')

In [ ]:
# Ćw. 6
def memoize(func):
    cache = {}
    def wrapper(*args):
        ...
    return wrapper


@memoize
def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)


print(fib(30))   # 832040

---

## 3. 🔹 `functools.reduce` i `total_ordering`

`functools.reduce(func, iterable, initial)` redukuje sekwencję do
pojedynczej wartości, wielokrotnie stosując `func(acc, element)`.

```
reduce(f, [a,b,c,d]) = f(f(f(a,b),c),d)
```

`@functools.total_ordering` uzupełnia klasę w metody porównań.
Wystarczy zdefiniować `__eq__` i jeden z: `__lt__`, `__le__`,
`__gt__`, `__ge__` - resztę dekorator wygeneruje automatycznie.

| Funkcja | Opis |
|---|---|
| `reduce(f, it)` | `f(f(f(a,b),c),d)` |
| `reduce(f, it, init)` | zaczyna od `init` |
| `@total_ordering` | generuje brakujące `<`,`>`,`<=`,`>=` |
| `wraps(func)` | kopiuje metadane funkcji dekorowanej |

In [ ]:
from functools import reduce, total_ordering

# reduce - iloczyn
numbers = [1, 2, 3, 4, 5]
product = reduce(lambda acc, x: acc * x, numbers, 1)
print(f'product: {product}')  # 120

# reduce - spłaszczenie listy list
nested = [[1, 2], [3, 4], [5, 6]]
flat = reduce(lambda acc, x: acc + x, nested, [])
print(f'flat: {flat}')  # [1,2,3,4,5,6]


@total_ordering
class Version:
    """Wersja semantyczna major.minor.patch."""

    def __init__(self, version: str):
        parts = version.split('.')
        self.major, self.minor, self.patch = map(int, parts)

    def __eq__(self, other) -> bool:
        return (self.major, self.minor, self.patch) == \
               (other.major, other.minor, other.patch)

    def __lt__(self, other) -> bool:
        return (self.major, self.minor, self.patch) < \
               (other.major, other.minor, other.patch)

    def __repr__(self) -> str:
        return f'{self.major}.{self.minor}.{self.patch}'


v1 = Version('1.2.3')
v2 = Version('1.3.0')
print(v1 < v2, v1 > v2, v1 <= v2)   # True False True
print(sorted([Version('2.0.0'), Version('1.9.9'), Version('1.0.0')]))

---

### 🐍 Ćwiczenia - reduce i total_ordering

**Ćw. 7.** Użyj `reduce`, aby znaleźć największy element z listy
liczb `[3, 1, 4, 1, 5, 9, 2, 6]` bez użycia `max()`.
Następnie oblicz sumę kwadratów tej listy.

**Ćw. 8.** Użyj `reduce`, aby scalić listę słowników w jeden:
`[{'a':1},{'b':2},{'c':3}]` → `{'a':1,'b':2,'c':3}`.

**Ćw. 9.** *(Trudniejsze)* Utwórz klasę `Card(suit, rank)`,
gdzie `rank` to liczba 2-14 (14=Ace), a `suit` to kolor.
Użyj `@total_ordering` i zdefiniuj tylko `__eq__` i `__lt__`
(porównanie tylko po ranku). Posortuj talię kart.
# hint: 'Ace' odpowiada rankowi 14

In [ ]:
# Ćw. 7
from functools import reduce

numbers = [3, 1, 4, 1, 5, 9, 2, 6]
maximum = ...
print(f'max: {maximum}')   # 9

sum_of_squares = ...
print(f'sum of squares: {sum_of_squares}')   # 183

In [ ]:
# Ćw. 8
from functools import reduce

dicts = [{'a': 1}, {'b': 2}, {'c': 3}]
merged = ...
print(merged)   # {'a': 1, 'b': 2, 'c': 3}

In [ ]:
# Ćw. 9
from functools import total_ordering

@total_ordering
class Card:
    SUITS = ['♣', '♦', '♥', '♠']
    RANKS = {2:'2',3:'3',4:'4',5:'5',6:'6',7:'7',8:'8',
             9:'9',10:'10',11:'J',12:'Q',13:'K',14:'A'}

    def __init__(self, suit: str, rank: int):
        ...

    def __eq__(self, other) -> bool:
        ...

    def __lt__(self, other) -> bool:
        ...

    def __repr__(self) -> str:
        return f'{self.RANKS[self.rank]}{self.suit}'


hand = [Card('♠', 10), Card('♥', 14), Card('♣', 5), Card('♦', 10)]
print(sorted(hand))